# Reproducing Tables 3 and 4

This notebook reproduces the upper-bound examples in Appendix B of the final paper.

- Appendix B, Example 2 gives the multi-class regular-demand upper bounds reported in Table 3.
- The one-class examples give the concrete performance ratios in Table 4; these ratios also provide the corresponding one-class upper bounds summarized in Table 3.
- The companion file reusable_pricing.py must be in the same directory as this notebook.
- Run all cells from top to bottom. The final cell runs every required example once and saves the detailed one-class results to table3_4_results.csv.


In [ ]:
"""Reproduce Tables 3 and 4 in Appendix B of the final paper."""

import csv
import numpy as np

from reusable_pricing import (
    compare_with_inventory_benchmark,
    exponential_demand,
    linear_demand,
    shifted_equal_revenue_demand,
    solve_inventory_policy_iteration,
    solve_one_class_probability,
    two_class_policy_matrix,
)

In [ ]:

def run_multiclass_example_2() -> None:
    C = 3
    demands = [
        linear_demand(180.0, 0.05, name="class 1"),
        linear_demand(11.0, 50.0, name="class 2"),
    ]
    service_rates = np.array([0.001, 1000.0])

    # Policy iteration is used here because the extreme service-rate ratio
    # makes uniformized relative value iteration unnecessarily slow.
    inventory = solve_inventory_policy_iteration(C, demands, service_rates)
    comparison = compare_with_inventory_benchmark(
        C,
        demands,
        service_rates,
        inventory,
    )

    print("Appendix B, Example 2 (Table 3)")
    print("Optimal class-1 arrival-rate matrix")
    print(two_class_policy_matrix(C, inventory, 0))
    print("Optimal class-2 arrival-rate matrix")
    print(two_class_policy_matrix(C, inventory, 1))
    print(f"Optimal inventory-based revenue: {inventory['revenue']:.8f}")
    print(f"tilde lambda: {comparison['tilde_rates']}")
    print(f"Revenue of tilde lambda: {comparison['tilde_revenue']:.8f}")
    print(f"Ratio for tilde lambda: {comparison['tilde_ratio']:.6%}")
    print(
        "Optimal static lambda:",
        comparison["optimal_static"]["arrival_rates"],
    )
    print(
        "Optimal static revenue:",
        f"{comparison['optimal_static']['revenue']:.8f}",
    )
    print(
        "Optimal static ratio:",
        f"{comparison['optimal_static_ratio']:.6%}",
    )
    print()




In [ ]:
def run_one_class_table_4(output_file: str = "table3_4_results.csv") -> None:
    examples = [
        {
            "label": "regular, C=3, tilde, mu approximately 0",
            "C": 3,
            "demand": shifted_equal_revenue_demand(10.0, shift=3.0),
            "mu": 1e-5,
        },
        {
            "label": "regular, C=3, optimal static",
            "C": 3,
            "demand": shifted_equal_revenue_demand(96.0, shift=30.5),
            "mu": 0.01,
        },
        {
            "label": "MHR, C=22, lambda^g and optimal static",
            "C": 22,
            "demand": exponential_demand(25.0),
            "mu": 0.1115,
        },
        {
            "label": "MHR, C=2, lambda^g and optimal static",
            "C": 2,
            "demand": exponential_demand(10.0),
            "mu": 0.7117,
        },
        {
            "label": "uniform, C=2, lambda^g and optimal static",
            "C": 2,
            "demand": linear_demand(6.0, 1.0, market_size=6.0),
            "mu": 1.0,
        },
    ]

    rows = []
    for example in examples:
        inventory = solve_one_class_probability(
            example["C"],
            example["demand"],
            example["mu"],
        )
        comparison = compare_with_inventory_benchmark(
            example["C"],
            [example["demand"]],
            np.array([example["mu"]]),
            inventory,
        )
        rows.append(
            {
                "example": example["label"],
                "C": example["C"],
                "Lambda": example["demand"]["Lambda"],
                "mu": example["mu"],
                "inventory_revenue": comparison["inventory_revenue"],
                "tilde_lambda": comparison["tilde_rates"][0],
                "tilde_ratio": comparison["tilde_ratio"],
                "lambda_g": comparison["geometric_rate"],
                "lambda_g_ratio": comparison["geometric_ratio"],
                "optimal_static_lambda": comparison["optimal_static"]["arrival_rates"][0],
                "optimal_static_ratio": comparison["optimal_static_ratio"],
            }
        )

    with open(output_file, "w", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    print("Appendix B, Table 4 (1-class examples)")
    for row in rows:
        print(
            f"{row['example']}: "
            f"tilde={np.ceil(row['tilde_ratio'] * 1_000_000) / 1_000_000:.4%}, "
            f"lambda^g={np.ceil(row['lambda_g_ratio'] * 1_000_000) / 1_000_000:.4%}, "
            f"optimal static={np.ceil(row['optimal_static_ratio'] * 1_000_000) / 1_000_000:.4%}"
        )
    print(f"Saved detailed results to {output_file}")


In [ ]:
# Run each required computation exactly once.
run_multiclass_example_2()
run_one_class_table_4()
